# 🎙️ DubFlow Studio — Free Cloud GPU Launcher
### High-Performance AI Video Dubbing with NVIDIA CUDA Acceleration (16GB T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jayendramanek/DubFlow-Studio/blob/main/DubFlow_Studio_Colab_Kaggle.ipynb)

This ready-to-run notebook runs **DubFlow Studio** completely free on **Google Colab** or **Kaggle** with full NVIDIA CUDA acceleration.

---

### ⚙️ Quick Environment Setup:
* **On Google Colab**:
  1. Top menu: **Runtime** ➔ **Change runtime type**.
  2. Select **T4 GPU** under *Hardware accelerator* and click **Save**.
* **On Kaggle**:
  1. Right sidebar under **Notebook options**:
  2. Set **Accelerator** to **GPU T4 x2** (or GPU P100).
  3. Set **Internet** to **On** *(requires one-time phone verification in Kaggle settings)*.

---
### 🚀 How It Works:
1. **Step 1**: Verifies your GPU and CUDA readiness.
2. **Step 2**: Clones the DubFlow repository & installs required AI models.
3. **Step 3**: Starts the secure Cloudflare tunnel and launches the FastAPI Web UI.
4. **Click the generated link** to open the full DubFlow Studio interface in your browser!


In [ ]:
# Step 1: Verify Hardware & GPU Acceleration
import os
import sys
import torch

print("=" * 70)
print("🔍 HARDWARE ACCELERATION CHECK")
print("=" * 70)

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    cuda_version = torch.version.cuda
    print(f"✅ NVIDIA GPU Detected  : {device_name}")
    print(f"✅ Dedicated VRAM       : {vram_gb:.2f} GB")
    print(f"✅ CUDA Toolkit Version : {cuda_version}")
    print("🚀 High-speed CUDA acceleration will be used for Demucs and Whisper!")
else:
    print("⚠️ WARNING: No NVIDIA GPU detected! Currently running on CPU.")
    print("To enable 10x-20x faster processing:")
    print("  • Google Colab: Runtime -> Change runtime type -> Select T4 GPU")
    print("  • Kaggle: Right Panel -> Accelerator -> Select GPU T4 x2 & Internet ON")
print("=" * 70)


In [ ]:
# Step 2: Clone DubFlow Studio & Install Dependencies
import os
import subprocess
import shutil

print("📦 Setting up DubFlow Studio environment...")

# 1. Clean up any broken/empty clone and clone repository fresh
target_dir = os.path.abspath("DubFlow-Studio")
if not os.path.exists(os.path.join(target_dir, "api.py")):
    print("Cloning fresh repository from GitHub...")
    !rm -rf "{target_dir}"
    !git clone https://github.com/jayendramanek/DubFlow-Studio.git "{target_dir}"
    %cd "{target_dir}"
else:
    %cd "{target_dir}"
    print("Updating existing repository...")
    !git pull origin main 2>/dev/null || true

# Verify api.py exists
!ls -lh api.py

# 2. Verify or install FFmpeg
ffmpeg_path = shutil.which("ffmpeg")
if not ffmpeg_path:
    print("Installing FFmpeg...")
    !apt-get update -qq && apt-get install -y -qq ffmpeg
print(f"✅ FFmpeg ready at: {shutil.which('ffmpeg')}")

# 3. Install Python dependencies
print("Installing Python packages...")
!pip install -q --upgrade pip
!pip install -q "av<18" nvidia-cublas-cu12 nvidia-cudnn-cu12 demucs yt-dlp faster-whisper deep-translator edge-tts fastapi "uvicorn[standard]" python-multipart psutil

# 4. Download Cloudflare Tunnel binary for remote Web UI access
if not os.path.exists("cloudflared"):
    print("Downloading Cloudflare Tunnel binary...")
    !curl -sL --output cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

print("\n✅ All dependencies successfully installed! Proceed to Step 3.")


In [ ]:
# Step 3: Launch Cloudflare Tunnel & DubFlow Studio Web Server
import os
import sys
import time
import re
import subprocess
from IPython.display import display, HTML

# 1. Clean up any existing instances
!pkill -f cloudflared 2>/dev/null || true
!pkill -f "python api.py" 2>/dev/null || true
time.sleep(1)

import shutil
# Ensure cloudflared binary exists and is executable
cf_path = shutil.which("cloudflared") or "/usr/local/bin/cloudflared"
if not os.path.exists(cf_path):
    for cand in ["./cloudflared", "/kaggle/working/DubFlow-Studio/cloudflared"]:
        if os.path.exists(cand):
            cf_path = cand
            break
if not os.path.exists(cf_path):
    print("Downloading cloudflared binary...")
    !curl -sL --output /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared
    cf_path = "/usr/local/bin/cloudflared"

# Ensure we are in DubFlow-Studio project directory
if os.path.exists("/kaggle/working/DubFlow-Studio"):
    %cd /kaggle/working/DubFlow-Studio
elif os.path.exists("/content/DubFlow-Studio"):
    %cd /content/DubFlow-Studio

# 2. Start Cloudflare Tunnel in background
print("🌐 Starting secure Cloudflare Tunnel...")
cf_proc = subprocess.Popen(
    [cf_path, "tunnel", "--url", "http://localhost:3333"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Read tunnel logs to extract public URL
public_url = None
timeout_seconds = 25
start_time = time.time()

for line in iter(cf_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+.trycloudflare.com', line)
    if match:
        public_url = match.group(0)
        break
    if time.time() - start_time > timeout_seconds:
        break

print("=" * 72)
if public_url:
    print(f"🎉 DUBFLOW STUDIO IS READY!")
    print(f"👉 Open Web UI: {public_url}")
    print("=" * 72)
    display(HTML(f'''
    <div style="background: linear-gradient(135deg, #111827, #1f2937); padding: 22px; border-radius: 12px; color: #fff; text-align: center; margin: 15px 0; border: 1px solid #374151; box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.5);">
        <h2 style="margin: 0 0 8px 0; color: #38bdf8; font-family: sans-serif;">🎙️ DubFlow Studio Web UI</h2>
        <p style="margin: 0 0 16px 0; font-size: 14px; color: #9ca3af; font-family: sans-serif;">Click the button below to access your dashboard with NVIDIA GPU acceleration:</p>
        <a href="{public_url}" target="_blank" style="background: #0284c7; color: #ffffff; padding: 12px 28px; border-radius: 8px; text-decoration: none; font-weight: 700; font-size: 16px; font-family: sans-serif; display: inline-block;">
            🚀 Open DubFlow Studio Dashboard
        </a>
        <p style="margin: 14px 0 0 0; font-size: 12px; color: #64748b; font-family: sans-serif;">Public Link: <a href="{public_url}" target="_blank" style="color: #38bdf8;">{public_url}</a></p>
    </div>
    '''))
else:
    print("⚠️ Cloudflare tunnel URL timed out. Server will start on http://localhost:3333")
    print("=" * 72)

# 4. Configure CUDA dynamic library path and launch FastAPI Server
os.environ["HOST"] = "0.0.0.0"
os.environ["PORT"] = "3333"
!LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$(python -c 'import site; print(":".join([p + "/nvidia/cublas/lib:" + p + "/nvidia/cudnn/lib" for p in site.getsitepackages()]))') python api.py


### 💾 Optional: Save Dubbed Videos Directly to Google Drive (Colab Only)
If you want completed videos automatically backed up or easily accessible on Google Drive:


In [ ]:
# (Optional - Google Colab Only) Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    export_dir = "/content/drive/MyDrive/DubFlow_Exports"
    os.makedirs(export_dir, exist_ok=True)
    print(f"✅ Google Drive mounted! Completed exports will be synced to: {export_dir}")
except Exception as e:
    print("ℹ️ Drive mount skipped (only applies to Google Colab).")
